In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "/Workspace/FMCG - Atlikon/consolidated_pipeline/01_Setup/Utilities"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
# Creating Widgets
dbutils.widgets.text("catalog","FMCG","Catalog")
dbutils.widgets.text("data_source","Products","Data_Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
base_path = f"s3://learning-atlikon-sportsbar/products/products.csv"
print(base_path)

# Bronze

In [0]:
df = (
    spark.read.format("csv")
    .option("header",True)
    .option("inferschema",True)
    .load(base_path)
    .withColumn("read_timestamp",F.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
)

In [0]:
df.printSchema()

In [0]:
display(df.limit(5))

In [0]:
df.write\
  .format("delta")\
  .option("delta.enableChangeDataFeed","true")\
  .mode("overwrite")\
  .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

# Silver

In [0]:
df_silver = spark.sql( f"select * from {catalog}.{bronze_schema}.{data_source};")
df_silver.show(5)

## 1.Drop Duplicates

In [0]:
print("Before duplicate removal",df_silver.count())
df_silver = df_silver.drop_duplicates(['product_id'])
print("After duplicate removal",df_silver.count())

## 2.Title Case fix

In [0]:
df_silver.select("category").distinct().show()

In [0]:
df_silver=df_silver.withColumn(
    "category",
    F.when(F.col("category").isNull(),None)
     .when(F.col("category")=="protien bars","Protein Bars")
     .otherwise(F.initcap("category"))
)\
    .withColumn(
    "product_name",
    F.regexp_replace(F.col("product_name"),"(?i)protien","Protein")
)
df_silver.select("category").distinct().show()

In [0]:
display(df_silver.limit(5))

## Standardizing Customer Attributes to match Parent Company Data Model

Note: This is something like Product manager given for Divison Column

In [0]:
df_silver = (df_silver.withColumn(
    "division",
    F.when(F.col("category")== "Energy Bars","Nutrition Bars")
     .when(F.col("category")== "Protein Bars","Nutrition Bars")
     .when(F.col("category")== "Granola & Cereals","Breakfast Foods")
     .when(F.col("category")== "Recovery Dairy","Dairy & Recovery")
     .when(F.col("category")== "Healthy Snacks","Healthy Snacks")
     .when(F.col("category")== "Electrolyte Mix","Hydration & Electrolytes")
     .otherwise("Others")
    )
)

In [0]:
# Separating Variant from the producvt name

df_silver = df_silver.withColumn(
    "Variant",
    F.regexp_extract(F.col("product_name"),r"\((.*?)\)",1)
)


In [0]:
display(df_silver.limit(5))

In [0]:
# Creating New Column Product Code:

df_silver =(
    df_silver.withColumn(
    "product_code",
    F.sha2(F.col("product_name").cast("string"),256)
)
# Clean Product ID keeing onjly numeric else replace with 9999
.withColumn(
    "product_id",
    F.when(
        F.col("product_id").cast("string").rlike("^[0-9]+$"),
        F.col("product_id").cast("string")
    ).otherwise(
        F.lit("9999").cast("string")
    )
)
#3.Rename ProductName to Product
.withColumnRenamed(
    "product_name","product"
)
)


In [0]:
df_silver = df_silver.select("product_code","division","category","product","Variant","product_id","read_timestamp","file_name","file_size")

In [0]:
display(df_silver)

In [0]:
df_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .option("delta.enableChangeDataFeed","true")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")


## Gold

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
df_gold = df_silver.select("product_code","product_id","division","category","product","Variant")
df_gold.show(5)

In [0]:
df_gold.write\
    .format("delta")\
    .option("delts.enableChangedataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")
